# Making observation plan

### Install modules

In [2]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, astroquery, requests, astropy, astropy.units, certifi, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")

### Import modules

In [3]:
from datetime import datetime, timedelta, timezone
from astropy.time import Time

import pandas as pd

import json
import requests

from astroquery.mpc import MPC
from pprint import pprint

from astropy.coordinates import SkyCoord
from astropy import units as u

### Set time

In [4]:
dt_kst_now = str(datetime.now())     # KST
start_dt_kst_str = dt_kst_now.replace(' ','T')[:19]    # observation start time KST
start_dt_kst = datetime.strptime(start_dt_kst_str, '%Y-%m-%dT%H:%M:%S')

start_dt_utc = start_dt_kst + timedelta(hours=-9)
start_dt_utc_str = start_dt_utc.strftime('%Y-%m-%dT%H:%M:%S')

### Set condition

In [5]:
mpc_code='P64'
obs_dt_utc = datetime.strptime(start_dt_utc_str, '%Y-%m-%dT%H:%M:%S')
obs_datetime = obs_dt_utc.strftime('%Y-%m-%dT%H:%M:%S')
elev_min=10
time_min=15
vmag_min=0
vmag_max=20
max_output=150
sort='trans'

# Define API URL and SPK filename:
url = f"https://ssd-api.jpl.nasa.gov/sbwobs.api?sb-kind=a&mpc-code={mpc_code}&obs-time={str(obs_datetime)}&elev-min={elev_min}&time-min={time_min}&vmag-max={vmag_max}&vmag-min={vmag_min}&optical=true&fmt-ra-dec=true&mag-required=true&output-sort={sort}&maxoutput={max_output}"
spk_filename = 'spk_file.bsp'

# Submit the API request and decode the JSON-response:
response = requests.get(url)
data = json.loads(response.text)

df = pd.DataFrame.from_dict(data['data'])
df = df.set_axis((data['fields']), axis=1)

### Current data

In [6]:
data_set=[]

for data_line in data['data']:
    name = data_line[1].split('(')[1][:-1]
    try:
        result = MPC.get_ephemeris(name, location = mpc_code)[0]

        c = SkyCoord(ra = result[1]*u.degree, dec = result[2]*u.degree)
        Ra,Dec = c.to_string('hmsdms').split(' ')
        Ra = Ra.replace('h',':').replace('m',':').replace('s','')
        Dec = Dec.replace('d',':').replace('m',':').replace('s','').replace('+','')

        Vmag = result[7]
        Azi = result[10]
        Alt = result[11]

        if int(Alt)>=elev_min:
            data_set.append([name, Ra, Dec, Vmag, Azi, Alt])
    except:
        continue

df1 = pd.DataFrame.from_dict(data_set)
df1 = df1.set_axis(['Name', 'RA', 'Dec', 'V-mag', 'Azimuth', 'Altitude'], axis=1)

### Xml document

In [7]:
ObsPlan = [[30,'r',1,10],[60,'r',1,10],[90,'r',1,10],[30,'r',2,10],[60,'r',2,10],[90,'r',2,10],]   # ExpTime, Filter, Bin, TotalExpCnt
Sequence_set = []

for asteroid_data in data_set:
  Targetname = asteroid_data[0].replace(' ','') + '_' + obs_datetime.replace('T','_').replace('-','').replace(':','')[2:-2]    # 제목 형식: Name_yymmdd_hhmm(UTC)
  
  RAHours, RAMinutes, RASeconds = asteroid_data[1].split(':');RASeconds = RASeconds[:2]   # hh, mm, ss
  DecDegrees, DecMinutes, DecSeconds = asteroid_data[2].split(':');DecSeconds = DecSeconds[:2]    # hh, mm, ss

  pre = """<?xml version="1.0" encoding="utf-8"?>\n"""
  CSlist = f"""<CaptureSequenceList xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" TargetName="{Targetname}" Mode="ROTATE" RAHours="{RAHours}" RAMinutes="{RAMinutes}" RASeconds="{RASeconds}" DecDegrees="{DecDegrees}" DecMinutes="{DecMinutes}" DecSeconds="{DecSeconds}" Rotation="0" Delay="0" SlewToTarget="true" AutoFocusOnStart="true" CenterTarget="true" RotateTarget="false" StartGuiding="true" AutoFocusOnFilterChange="false" AutoFocusAfterSetTime="false" AutoFocusSetTime="30" AutoFocusAfterSetExposures="false" AutoFocusSetExposures="10" AutoFocusAfterTemperatureChange="false" AutoFocusAfterTemperatureChangeAmount="5" AutoFocusAfterHFRChange="false" AutoFocusAfterHFRChangeAmount="10">\n"""
  xmlDoc = pre + CSlist   # xml 문서

  for Obs in ObsPlan:   # 관측 계획(알짜 요소)
    CapSeq = f"""  <CaptureSequence>
    <Enabled>true</Enabled>
    <ExposureTime>{Obs[0]}</ExposureTime>
    <ImageType>LIGHT</ImageType>
    <FilterType>
      <Name>{Obs[1]}</Name>
      <FocusOffset>0</FocusOffset>
      <Position>3</Position>
      <AutoFocusExposureTime>-1</AutoFocusExposureTime>
      <AutoFocusFilter>false</AutoFocusFilter>
      <FlatWizardFilterSettings>
        <FlatWizardMode>DYNAMICEXPOSURE</FlatWizardMode>
        <HistogramMeanTarget>0.5</HistogramMeanTarget>
        <HistogramTolerance>0.1</HistogramTolerance>
        <MaxFlatExposureTime>30</MaxFlatExposureTime>
        <MinFlatExposureTime>0.01</MinFlatExposureTime>
        <StepSize>0.1</StepSize>
        <MaxAbsoluteFlatDeviceBrightness>1</MaxAbsoluteFlatDeviceBrightness>
        <MinAbsoluteFlatDeviceBrightness>0</MinAbsoluteFlatDeviceBrightness>
        <FlatDeviceAbsoluteStepSize>1</FlatDeviceAbsoluteStepSize>
      </FlatWizardFilterSettings>
      <AutoFocusBinning>
        <X>1</X>
        <Y>1</Y>
      </AutoFocusBinning>
      <AutoFocusGain>-1</AutoFocusGain>
      <AutoFocusOffset>-1</AutoFocusOffset>
    </FilterType>
    <Binning>
      <X>{Obs[2]}</X>
      <Y>{Obs[2]}</Y>
    </Binning>
    <Gain>-1</Gain>
    <Offset>-1</Offset>
    <TotalExposureCount>{Obs[3]}</TotalExposureCount>
    <ProgressExposureCount>0</ProgressExposureCount>
    <Dither>true</Dither>
    <DitherAmount>1</DitherAmount>
  </CaptureSequence>\n"""
    xmlDoc += CapSeq

  end = f"""  <Coordinates>
    <RA>{int(RAHours) + int(RAMinutes)/60 + int(RASeconds)/3600}</RA>
    <Dec>{int(DecDegrees) + int(DecMinutes)/60 + int(DecMinutes)/3600}</Dec>
    <Epoch>J2000</Epoch>
  </Coordinates>
  <NegativeDec>false</NegativeDec>
</CaptureSequenceList>\n"""
  xmlDoc += end
  
  Sequence_set.append([Targetname, xmlDoc])   # 제목, xml 문서 내용


### Save xml document

In [10]:
path = "D:\Downloads\\230912\\FileName.xml"     # FileName 자리에 xml 문서 이름

for Sequence in Sequence_set:
    file = open(path.replace('FileName',Sequence[0]), 'w')
    file.write(Sequence[1])
    file.close